# Tópico 5 e 6: Variável-Alvo e Feature Engineering
Este notebook é responsável por:
1. Criar a variável-alvo (`quebra_safra`) baseada no limiar de rendimento (Tópico 5).
2. Remover variáveis que causariam Target Leakage (Vazamento de Dados).
3. Criar novas variáveis preditoras baseadas nos dados agroclimáticos (Tópico 6).

In [1]:
import pandas as pd
import os

# Carregar os dados brutos integrados
df_raw = pd.read_csv('../data/raw/integrado.csv')
display(df_raw.head())

,ano,area_plantada_ha,area_colhida_ha,quantidade_produzida_t,rendimento_kg_ha,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,GWETROOT
0,2015,400.0,400.0,960.0,2400.0,1292.21,22.402685,39.91,9.01,74.525507,0.589973
1,2016,500.0,500.0,1200.0,2400.0,1243.65,21.431339,37.37,3.24,76.412322,0.640492
2,2017,500.0,500.0,1200.0,2400.0,1252.76,21.700493,37.61,5.39,73.824438,0.601863
3,2018,500.0,500.0,1200.0,2400.0,1126.40,22.172630,36.06,5.82,72.083233,0.569863
4,2019,1200.0,1200.0,3000.0,2500.0,1056.55,22.776274,39.10,4.43,70.784301,0.567205


## Tópico 5: Definição da Variável-Alvo
- **Alvo (`quebra_safra`)**: Definido como `1` se `rendimento_kg_ha < 2450`, e `0` caso contrário (conforme `RFC.md`).
- **Target Leakage**: Variáveis que representam informações de "pós-colheita" não podem ser usadas para prever o risco antes da colheita. Por isso, excluímos: `rendimento_kg_ha`, `quantidade_produzida_t`, `area_colhida_ha`, `area_plantada_ha`.

In [2]:
df_target = df_raw.copy()

# 1. Cria a variável alvo
df_target['quebra_safra'] = (df_target['rendimento_kg_ha'] < 2450).astype(int)

# 2. Remoção das variáveis vazadas
colunas_vazamento = [
    'rendimento_kg_ha', 
    'quantidade_produzida_t', 
    'area_colhida_ha', 
    'area_plantada_ha'
]
df_target = df_target.drop(columns=colunas_vazamento)

print("Colunas após o Tópico 5:")
print(df_target.columns.tolist())
display(df_target[['ano', 'quebra_safra']].head())

Colunas após o Tópico 5:
['ano', 'PRECTOTCORR', 'T2M', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'GWETROOT', 'quebra_safra']


,ano,quebra_safra
0,2015,1
1,2016,1
2,2017,1
3,2018,1
4,2019,0


## Tópico 6: Criação de Novas Variáveis (Feature Engineering)
Vamos criar variáveis meteorológicas derivadas que possam explicar melhor o estresse sofrido pela planta:
1. **Amplitude Térmica (`amplitude_termica`)**: Variações muito grandes de temperatura no mesmo dia geram estresse hídrico e fisiológico.
2. **Índice de Precipitação por Temperatura (`precipitacao_por_temperatura`)**: Mede o balanço de água disponível frente à evaporação gerada pelo calor.
3. **Interação Umidade x Precipitação (`GWET_x_PREC`)**: Identifica anos onde choveu muito e o solo efetivamente reteve essa água.

In [3]:
df_features = df_target.copy()

# 1. Amplitude Térmica Diária Média
df_features['amplitude_termica'] = df_features['T2M_MAX'] - df_features['T2M_MIN']

# 2. Indice de Estresse Hídrico Simples (Chuva / Temperatura Média)
df_features['precipitacao_por_temperatura'] = df_features['PRECTOTCORR'] / (df_features['T2M'] + 0.001)

# 3. Interação: Umidade do Solo x Precipitação
df_features['GWET_x_PREC'] = df_features['GWETROOT'] * df_features['PRECTOTCORR']

display(df_features.head())

,ano,PRECTOTCORR,T2M,T2M_MAX,T2M_MIN,RH2M,GWETROOT,quebra_safra,amplitude_termica,precipitacao_por_temperatura,GWET_x_PREC
0,2015,1292.21,22.402685,39.91,9.01,74.525507,0.589973,1,30.90,57.678458,762.368497
1,2016,1243.65,21.431339,37.37,3.24,76.412322,0.640492,1,34.13,58.026798,796.547631
2,2017,1252.76,21.700493,37.61,5.39,73.824438,0.601863,1,32.22,57.726903,753.989909
3,2018,1126.40,22.172630,36.06,5.82,72.083233,0.569863,1,30.24,50.799079,641.893699
4,2019,1056.55,22.776274,39.10,4.43,70.784301,0.567205,0,34.67,46.386148,599.280949


## Salvando os dados processados
Salvaremos na pasta `data/interim/` para que os próximos passos do projeto possam consumi-los.

In [4]:
# Criar diretório se não existir
os.makedirs('../data/interim', exist_ok=True)

# Salvar o dataset com features
path_out = '../data/interim/04_dados_com_features.csv'
df_features.to_csv(path_out, index=False)
print(f"Dados salvos com sucesso em: {path_out}")

Dados salvos com sucesso em: ../data/interim/04_dados_com_features.csv
